In [47]:
# MIMIC-IV
from pathlib import Path

import json
import polars as pl
import time
import datetime

In [48]:
# Paths
## Directories
data_dir = Path("/workspaces/example/data")
shared_dir = Path("/workspaces/OpenICU-Example/custom/shared")

## Files
ricu_cnpt_dict_path = data_dir / "projects" / "ricu" / "inst" / "extdata" / "config" /  "concept-dict.json"
ricu_data_src_path = data_dir / "projects" / "ricu" / "inst" / "extdata" / "config" / "data-sources.json"

d_items_path = data_dir / "physionet.org" / "files" / "mimiciv" / "3.1" / "icu" / "d_items.csv.gz"
d_labitems_path = data_dir / "physionet.org" / "files" / "mimiciv" / "3.1" / "hosp" / "d_labitems.csv.gz"

In [49]:
# Json, dataframes and lazyframes

ricu_cnpt_dict_json = None
ricu_data_src_json = None

d_items_lf = None
d_labitems_lf = None

In [50]:
# ricu concept dict

ricu_cnpt_dict_df = pl.read_json(ricu_cnpt_dict_path)
with open(ricu_data_src_path, "r", encoding="utf-8") as f:
    ricu_data_src_data = json.load(f)
# miiv data
miiv_d_items_lf = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/icu/d_items.csv.gz").with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))
miiv_d_labitems_lf = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/hosp/d_labitems.csv.gz").with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))

In [51]:
for e in ricu_data_src_data:
    if e["name"] == "miiv":
        tables = e["tables"]



In [52]:
def ricu_cnpt_dict_to_cust_cnpt_dict(info: pl.DataFrame, src_name: str, src_items: dict[str, pl.LazyFrame], src_item_col_name: str) -> tuple[dict, dict]:
    data: dict[str, dict] = {}
    un_src_data: dict = {}
    i = 0
    for cnpt in info.columns:
        if(cnpt_srcs := info[cnpt][0].get("sources")) and cnpt_srcs.get(src_name):
            for i in range(len(cnpt_srcs[src_name])):
                src = cnpt_srcs[src_name][i]
                cnpt_ids = src.get("ids")
                cnpt_regex = src.get("regex")
                if cnpt_ids is None and cnpt_regex is None:
                    un_src_data[cnpt] = info[cnpt]
                    continue
                cnpt_entr = info[cnpt][0]
                cnpt_cat = cnpt_entr["category"].replace(" ", "_").replace("/", "_per_")
                cnpt_name = cnpt_entr["description"].replace(" ", "_").replace("/", "_per_")
                cnpt_tbl = src["table"]
                sub_var = src["sub_var"] 

                if not data.get(cnpt_cat):
                    data[cnpt_cat] = {}

                if isinstance(cnpt_ids, str) or (isinstance(cnpt_ids, list) and isinstance(cnpt_ids[0], str)):
                    un_src_data[cnpt] = info[cnpt]
                    continue
                
                if cnpt_ids is not None:
                    cnpt_ids: list[int] = cnpt_ids if isinstance(cnpt_ids, list) else [cnpt_ids]

                if data[cnpt_cat].get(cnpt_name) is None:
                    data[cnpt_cat][cnpt_name] = []

                if isinstance((time_vars := tables[cnpt_tbl]["defaults"]["time_vars"]), list):
                    time = time_vars[0]
                else:

                    time = time_vars 
                
                data[cnpt_cat][cnpt_name].append({
                    "name": cnpt_name,
                    "unit": cnpt_entr.get("unit"),
                    "table": cnpt_tbl,
                    "code": None,
                    "ids": cnpt_ids,
                    "src_regex": cnpt_regex,
                    "short": cnpt,
                    "min": cnpt_entr.get("min"),
                    "max": cnpt_entr.get("max"),
                    "sub_var": sub_var,
                    "time": time,
                })
                if cnpt_ids is not None:
                    filtered_src_items = src_items[cnpt_tbl if cnpt_tbl in src_items.keys() else "else"].filter(pl.col(src_item_col_name).is_in(cnpt_ids)).select("code").collect()
                    regex = "|".join(list(filtered_src_items["code"]))
                else:
                    regex = cnpt_regex

                data[cnpt_cat][cnpt_name][i]["code"] = "(" + regex + ")"

                if data[cnpt_cat][cnpt_name][i]["max"] is None:
                    data[cnpt_cat][cnpt_name][i]["max"] = "Inf"

                if data[cnpt_cat][cnpt_name][i]["min"] is None:
                    data[cnpt_cat][cnpt_name][i]["min"] = "-Inf"

    return (data, un_src_data)

In [53]:
data_by_cat = {}
un_src_data = {}

src_item_mapping = {"labevents": miiv_d_labitems_lf, "else": miiv_d_items_lf}
(data_by_cat, un_src_data) = ricu_cnpt_dict_to_cust_cnpt_dict(ricu_cnpt_dict_df, "miiv", src_item_mapping, "itemid")

In [54]:
data_by_cnpt = {}
for cat in data_by_cat.keys():
    for cnpt in data_by_cat[cat]:
        data_by_cnpt[cnpt] = data_by_cat[cat][cnpt]

In [55]:
# Overview about usable data
print("total category count:", len(data_by_cat))
for key in data_by_cat.keys():
    print("\tcategory:", key, "concept count:", len(data_by_cat[key]))
print("total concept count:", len(data_by_cnpt))

total category count: 9
	category: vitals concept count: 6
	category: chemistry concept count: 21
	category: hematology concept count: 20
	category: medications concept count: 13
	category: output concept count: 1
	category: blood_gas concept count: 9
	category: neurological concept count: 4
	category: respiratory concept count: 4
	category: demographics concept count: 2
total concept count: 80


In [56]:
# Overview about unused data
print("total unused concept count:", len(un_src_data))


total unused concept count: 8


In [57]:
with open(f"/workspaces//OpenICU-Example//custom//shared//check-open_icu-to-ricu-{datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S')}.json", "w") as file:
    json.dump(data_by_cat, file, indent=4, ensure_ascii=False)

In [58]:
def create_general_cnpt(dir_path: str | Path, name: str, unit: str, table_contents: list[dict]) -> None:
    if isinstance(unit, list):
        unit = unit[0]
    if unit == "%":
        unit = "\"%\""

    path = Path(dir_path)
    path.mkdir(parents=True, exist_ok=True)
    print(path)
    out_file = path / f"{name}.yml"

    # Existenz prüfen
    if out_file.exists():
        print(f"File already exists: {out_file}")
        return

    content = f"""name: {name}
version: 1.0.0
unit: {unit}
extension_columns:
  dataset: col("dataset")
  table: col("table")
"""

    with open(out_file, "w", encoding="utf8") as f:
        f.write(content)

    print(f"Created: {out_file}")

In [59]:
def create_src_specific_cnpt(dir_path: str | Path, name: str, unit: str, table_contents: list[dict]) -> None:
    if isinstance(unit, list):
        unit = unit[0]
    if unit == "%":
        unit = "\"%\""

    path = Path(dir_path)
    path.mkdir(parents=True, exist_ok=True)
    print(path)
    out_file = path / f"{name}.yml"

    # Existenz prüfen
    if out_file.exists():
        print(f"File already exists: {out_file}")
        return

    content = """mappings:
"""
    for table_content in table_contents:
      content +=  f"""  - pattern:
      table: {table_content["table"]}
      event: result
      code: {table_content["code"]}
    columns:
      numeric_value: col(numeric_value)
      text_value: col(text_value)
"""

    with open(out_file, "w", encoding="utf8") as f:
        f.write(content)

    print(f"Created: {out_file}")

In [60]:
base_path = Path("/workspaces/OpenICU-Example/custom/generated/temp/config/mimic-iv/3.1/concept")

for category_name in data_by_cat.keys():
    # folder = base_path / category_name.replace(" ", "_").replace("/", "_per_")
    folder = base_path
    folder.mkdir(parents=True, exist_ok=True)
    for cnpt in (category := data_by_cat[category_name]).keys() :
        create_src_specific_cnpt(folder,
        category[cnpt][0]["name"].replace(" ", "_").replace("/", "_per_"),
        category[cnpt][0]["unit"] if isinstance(category[cnpt][0]["unit"], list) else category[cnpt][0]["unit"] ,
        [{"table": category[cnpt][i]["table"], "code": category[cnpt][i]["code"]} for i in range(len(category[cnpt]))]
        )
        create_general_cnpt(Path("/workspaces/OpenICU-Example/custom/generated/temp/config/concept") / category_name.replace(" ", "_").replace("/", "_per_"), category[cnpt][0]["name"].replace(" ", "_").replace("/", "_per_"),
        category[cnpt][0]["unit"] if isinstance(category[cnpt][0]["unit"], list) else category[cnpt][0]["unit"] ,
        [{"table": category[cnpt][i]["table"], "code": category[cnpt][i]["code"]} for i in range(len(category[cnpt]))])
    

/workspaces/OpenICU-Example/custom/generated/temp/config/mimic-iv/3.1/concept
File already exists: /workspaces/OpenICU-Example/custom/generated/temp/config/mimic-iv/3.1/concept/endtidal_CO2.yml
/workspaces/OpenICU-Example/custom/generated/temp/config/concept/vitals
File already exists: /workspaces/OpenICU-Example/custom/generated/temp/config/concept/vitals/endtidal_CO2.yml
/workspaces/OpenICU-Example/custom/generated/temp/config/mimic-iv/3.1/concept
File already exists: /workspaces/OpenICU-Example/custom/generated/temp/config/mimic-iv/3.1/concept/temperature.yml
/workspaces/OpenICU-Example/custom/generated/temp/config/concept/vitals
File already exists: /workspaces/OpenICU-Example/custom/generated/temp/config/concept/vitals/temperature.yml
/workspaces/OpenICU-Example/custom/generated/temp/config/mimic-iv/3.1/concept
File already exists: /workspaces/OpenICU-Example/custom/generated/temp/config/mimic-iv/3.1/concept/diastolic_blood_pressure.yml
/workspaces/OpenICU-Example/custom/generated

In [61]:
# Run pipeline.ipynb

# Sanity check

In [62]:
# Prepare needed information to make check inside of ricu/ R

# Root folder were the concepts are saved
cnpt_root_folder = Path("/workspaces/OpenICU-Example/output/project/workspace/concept")

found_data_info = {}

# iterate all the folders in the root folder
for cnpt_folder in cnpt_root_folder.iterdir():
    # if the cnpt_folder is a folder/directory execute logic
    if cnpt_folder.is_dir():
        # select the first file in the sub folder "/1.0.0/" and exprect it to be the cnpt_file
        cnpt_file  = next((file for file in (cnpt_folder / "1.0.0").iterdir() if file.is_file()), None)
        # print(f"{cnpt_folder}: {cnpt_file.name if cnpt_file else "empty"}")
        size = len(pl.read_parquet(cnpt_file))
        # print(size)
        name = cnpt_folder.name
        if data_by_cnpt.get(name):
            found_data_info[name] = data_by_cnpt[name]
        else:
            print("Not found: ", name)

FileNotFoundError: [Errno 2] No such file or directory: '/workspaces/OpenICU-Example/output/project/workspace/concept'

In [ ]:
found_data_info
data_by_cnpt

{'neutrophils': [{'name': 'neutrophils',
   'unit': '%',
   'table': 'labevents',
   'code': '(51256//Neutrophils)',
   'ids': [51256],
   'src_regex': None,
   'short': 'neut',
   'min': 0,
   'max': 100,
   'sub_var': 'itemid'}],
 'Hemoglobin_A1C': [{'name': 'Hemoglobin_A1C',
   'unit': '%',
   'table': 'labevents',
   'code': '(50852//% Hemoglobin A1c)',
   'ids': [50852],
   'src_regex': None,
   'short': 'hba1c',
   'min': '-Inf',
   'max': 'Inf',
   'sub_var': 'itemid'}],
 'basophils': [{'name': 'basophils',
   'unit': '%',
   'table': 'labevents',
   'code': '(51146//Basophils)',
   'ids': [51146],
   'src_regex': None,
   'short': 'basos',
   'min': 0,
   'max': 50,
   'sub_var': 'itemid'}],
 'eosinophils': [{'name': 'eosinophils',
   'unit': '%',
   'table': 'labevents',
   'code': '(51200//Eosinophils)',
   'ids': [51200],
   'src_regex': None,
   'short': 'eos',
   'min': 0,
   'max': 50,
   'sub_var': 'itemid'}],
 'erythrocyte_sedimentation_rate': [{'name': 'erythrocyte_sed

In [ ]:
with open("check-open_icu-to-ricu.json", "w") as file:
    json.dump(found_data_info, file, indent=4, ensure_ascii=False)
# found_data_info

## Make Checklist for Github

In [ ]:

def make_issue_checklist(ricu: pl.DataFrame):
    cat = {}

    for col in ricu.columns:
        elem = list(ricu[col])[0]
        if not cat.get(elem["category"]):
            cat[elem["category"]] = []
        cat[elem["category"]] += [elem["description"]]
    output = ""
    for cat, cnpts in cat.items():
        output += f"- [ ] {cat}:\n"
        for cnpt in cnpts:
            output += ((f"  - [x] {cnpt}\n") if (cnpt.replace(" ", "_").replace("/", "_per_") in data_by_cnpt.keys()) else (f"  - [ ] {cnpt}\n"))
            if (cnpt.replace(" ", "_").replace("/", "_per_") not in data_by_cnpt.keys()):
                print(f"{cnpt}")
    return output

In [ ]:
output = make_issue_checklist(ricu_cnpt_dict_df)

patient sex
patient body mass index
patient age
patient admission type
carboxyhemoglobin
sequential organ failure assessment score
national early warning score
in hospital mortality
dopamine administration for min 1h
SOFA central nervous system component
SOFA cardiovascular component
SOFA coagulation component
epinephrine administration for min 1h
SOFA renal component
SOFA liver component
SOFA respiratory component
modified early warning score
norepinephrine administration for min 1h
hospital length of stay
ICU length of stay
sepsis-3 criterion
quick SOFA score
systemic inflammatory response syndrome score
suspected infection
ventilation durations
Horowitz index
ventilation start
SaO2/FiO2
ventilation end
supplemental oxygen
tracheostomy
Glasgow coma scale (non-sedated)
AVPU scale
GCS total
urine output per 24h
corticosteroids
vasopressor indicator
dobutamine administration for min 1h
norepinephrine equivalents
body fluid sampling
